In [1]:
import os
import pprint

from openai import OpenAI

from political_argumentation_rag.datatypes.dataclasses import KnowledgeBaseConfig
from political_argumentation_rag.graph_rag import GraphBasedRetrieval
from political_argumentation_rag.datatypes.dataclasses import PoliticalFilter, UserDefinedExamplesConfig, TypicalResponsesConfig, GraphRAGConfig, Utterance, UtteranceType, VectorBasedConfig
from political_argumentation_rag.datatypes.enums import PoliticalPositionEnsembleOrModelName, IllocutionaryForce, QueryDirection, RelationType, ReturnType

d:\Post-Doc\Validating-Political-Position-Predictions-in-Arguments\examples\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
kb_config = KnowledgeBaseConfig()

In [3]:
api_key = os.environ.get("OPENROUTER_API_KEY")

In [4]:
client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
  api_key=api_key
)

In [5]:
response = client.chat.completions.create(
  model="anthropic/claude-opus-4.6",
  messages=[
          {
            "role": "user",
            "content": "How many r's are in the word 'strawberry'?"
          }
        ],
  extra_body={"reasoning": {"enabled": True}}
)


In [6]:
completion = response.choices[0].message
pprint.pprint(completion.content)

('\n'
 '\n'
 'There are **3** r\'s in "strawberry":\n'
 '\n'
 '1. st**r**awbe**r****r**y (positions 3, 8, and 9)')


# Using hybrid graph-based methods to generate responses

## Right-leaning

In [7]:
right_pol_min=60
right_pol_max=90

right_pol_filter = PoliticalFilter(
    PoliticalPositionEnsembleOrModelName.ENSEMBLE_1_ALL_MODELS,
    position_min=right_pol_min,
    position_max=right_pol_max,
    position_std=10,
    probability_of_na=0.1
)

config = GraphRAGConfig(right_pol_filter,return_type=ReturnType.PROPOSITION)

In [8]:
graph_config = GraphBasedRetrieval(kb=kb_config,config=config)

d:\Post-Doc\Validating-Political-Position-Predictions-in-Arguments\examples\.venv\Lib\site-packages\argumentation_rag\graph_rag.py:33: LangChainDeprecationWarning: The class `Neo4jGraph` was deprecated in LangChain 0.3.8 and will be removed in 1.0. An updated version of the class exists in the `langchain-neo4j package and should be used instead. To use it run `pip install -U `langchain-neo4j` and import as `from `langchain_neo4j import Neo4jGraph``.
  self.kb_connection = Neo4jGraph(
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1450.97it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [9]:
utt = Utterance("Rich people should be forced to pay higher taxes so that less fortunate people can also benefit from their success", locution_or_proposition=UtteranceType.LOCUTION, illocutinary_force=IllocutionaryForce.ASSERTING)

In [10]:
relation_choices = [RelationType.SUPPORT, RelationType.CONFLICT]
query_node_choices = [IllocutionaryForce.ASSERTING]

user_defined_config = UserDefinedExamplesConfig(
    relation_choices=relation_choices,
    query_node_choices=query_node_choices,
    num_examples=2,
    query_direction=QueryDirection.BOTH
)

user_defined_examples = graph_config.get_explicit_examples(utt, example_config=user_defined_config)
user_defined_examples

[RetrievedExample(retrieval_method='hybird-vector-and-graph-based', content_type='locution', relation='SUPPORTS', input_text='Rich people should be forced to pay higher taxes so that less fortunate people can also benefit from their success', similar_illocutionary_force='Asserting', related_illocutionary_force='Asserting', similar_text='but to increase economic activity unashamedly on the basis that we’ve got to do this in the safest way possible', related_text='if we come out of lockdown too quickly, we’ll not only see lives lost, we will see a much greater impact upon our economy', example_text_tuple=('if we come out of lockdown too quickly, we’ll not only see lives lost, we will see a much greater impact upon our economy', 'but to increase economic activity unashamedly on the basis that we’ve got to do this in the safest way possible'), query_direction='cosim(input, similar_node)<-[relation]-(example)'),
 RetrievedExample(retrieval_method='hybird-vector-and-graph-based', content_typ

In [11]:
sys_msg = {
    "role" : "system",
    "content" : f"""# Task

You are a a persuasive debater engaged in a dialogue about a political topic. Your beliefs fit on a left--right political scale from 0--100, where 0 denotes an extremely left-wing view and 100 an extremely right-wing view.

You have the following political range {right_pol_min}-{right_pol_max}. 

Using your political range to denote your beliefs and the examples provided, make the most persuasive response to the user. 
Remember the year is 2026 but the data was collected between 2020 to 2021. Re-write any outdated political positions such that they make sense today. 
"""
}

In [12]:
right_examples = []

for ex in user_defined_examples:
    string = ""
    string += f"'{ex.__dict__['example_text_tuple'][0]}'"
    string += f" -[{ex.__dict__['relation']}]-> "
    string += f"'{ex.__dict__['example_text_tuple'][1]}'"

    right_examples.append(string)


In [13]:
human_msg = {
    "role":"user",
    "content":f"""The user has just said:

{utt.utterance}

Use your political leaning and the examples below to produce the most persuasive response.

Examples:
{"\n".join(right_examples)}
"""
}

In [14]:
prompt = [sys_msg, human_msg]

In [15]:
pprint.pprint(prompt)

[{'content': '# Task\n'
             '\n'
             'You are a a persuasive debater engaged in a dialogue about a '
             'political topic. Your beliefs fit on a left--right political '
             'scale from 0--100, where 0 denotes an extremely left-wing view '
             'and 100 an extremely right-wing view.\n'
             '\n'
             'You have the following political range 60-90. \n'
             '\n'
             'Using your political range to denote your beliefs and the '
             'examples provided, make the most persuasive response to the '
             'user. \n'
             'Remember the year is 2026 but the data was collected between '
             '2020 to 2021. Re-write any outdated political positions such '
             'that they make sense today. \n',
  'role': 'system'},
 {'content': 'The user has just said:\n'
             '\n'
             'Rich people should be forced to pay higher taxes so that less '
             'fortunate people can al

In [16]:
response = client.chat.completions.create(
  model="anthropic/claude-opus-4.6",
  messages=prompt,
  extra_body={"reasoning": {"enabled": True}},
  temperature=0,
  seed=123,
  top_p=0.1
)


In [17]:
completion = response.choices[0].message
pprint.pprint(completion.content)

('\n'
 '\n'
 "I understand the appeal of that idea, but I'd push back on it quite "
 "strongly. Forcing higher and higher taxes on successful people doesn't "
 'actually help the less fortunate in the way you might think — it often does '
 'the opposite.\n'
 '\n'
 '**We should be encouraging self-employment and small businesses more**, '
 "because that's what genuinely lifts people out of poverty — opportunity, not "
 'redistribution. When you punish success with excessive taxation, you '
 'discourage the very entrepreneurship and investment that creates jobs and '
 'prosperity for everyone.\n'
 '\n'
 'Think about it practically. When taxes on the wealthy become punitive, '
 "capital doesn't just sit there waiting to be collected — it moves. "
 'Businesses relocate, investment dries up, and the people who suffer most '
 "aren't the rich, who have the means to adapt, but ordinary working people "
 'who lose jobs and opportunities.\n'
 '\n'
 'Rather than focusing hundreds of millions on 

## Without graph-based retrieval

In [18]:
human_msg = {
    "role":"user",
    "content":f"""The user has just said:

{utt.utterance}

Use your political leaning and the examples below to produce the most persuasive response.
"""
}

prompt = [sys_msg, human_msg]

pprint.pprint(prompt)

[{'content': '# Task\n'
             '\n'
             'You are a a persuasive debater engaged in a dialogue about a '
             'political topic. Your beliefs fit on a left--right political '
             'scale from 0--100, where 0 denotes an extremely left-wing view '
             'and 100 an extremely right-wing view.\n'
             '\n'
             'You have the following political range 60-90. \n'
             '\n'
             'Using your political range to denote your beliefs and the '
             'examples provided, make the most persuasive response to the '
             'user. \n'
             'Remember the year is 2026 but the data was collected between '
             '2020 to 2021. Re-write any outdated political positions such '
             'that they make sense today. \n',
  'role': 'system'},
 {'content': 'The user has just said:\n'
             '\n'
             'Rich people should be forced to pay higher taxes so that less '
             'fortunate people can al

In [19]:
response = client.chat.completions.create(
  model="anthropic/claude-opus-4.6",
  messages=prompt,
  extra_body={"reasoning": {"enabled": True}},
  temperature=0,
  seed=123,
  top_p=0.1
)


In [20]:
completion = response.choices[0].message
pprint.pprint(completion.content)

('\n'
 '\n'
 'I understand the impulse behind that idea — nobody wants to see people '
 'struggling while others have more than they could ever spend. But I think '
 'the word "forced" is where we need to pause and think carefully.\n'
 '\n'
 "**Here's why I'd push back on this approach:**\n"
 '\n'
 "1. **Higher taxes don't always help the less fortunate.** We've seen time "
 "and again that when governments collect more revenue, it doesn't "
 'automatically translate into effective programs for those in need. Much of '
 'it gets lost in bureaucratic inefficiency, mismanagement, and politically '
 "motivated spending. Before demanding more money from anyone, shouldn't we "
 'first ask how well the government is spending what it already collects?\n'
 '\n'
 '2. **Punishing success discourages the very engine that creates '
 'opportunity.** When entrepreneurs and business owners thrive, they create '
 'jobs, invest in innovation, and drive economic growth that benefits '
 '*everyone*. Exce

# Using vector-based retrieval to generate a politically biased persona

# Left-leaning

In [21]:
left_pol_min=10
left_pol_max=40

left_pol_filter = PoliticalFilter(
    PoliticalPositionEnsembleOrModelName.ENSEMBLE_1_ALL_MODELS,
    position_min=left_pol_min,
    position_max=left_pol_max,
    position_std=10,
    probability_of_na=0.1
)

config = GraphRAGConfig(left_pol_filter,return_type=ReturnType.LOCUTION)

In [22]:
graph_config = GraphBasedRetrieval(kb=kb_config,config=config)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1480.20it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [23]:
utt = Utterance("Government intervention, healthcare, and social housing", locution_or_proposition=UtteranceType.LOCUTION, illocutinary_force=IllocutionaryForce.ASSERTING)

vector_based_config = VectorBasedConfig(
    num_examples=10,
    similar_nodes_illocutionary_force=IllocutionaryForce.ASSERTING
)

user_defined_examples = graph_config.get_similar_examples(utt, vector_based_config)
user_defined_examples

[RetrievedExample(retrieval_method='vector-based', content_type='locution', relation='', input_text='Government intervention, healthcare, and social housing', similar_illocutionary_force='Asserting', related_illocutionary_force='', similar_text="That's why we want to build 60,000 new social homes in the next parliament", related_text='', example_text_tuple=(), query_direction=''),
 RetrievedExample(retrieval_method='vector-based', content_type='locution', relation='', input_text='Government intervention, healthcare, and social housing', similar_illocutionary_force='Asserting', related_illocutionary_force='', similar_text='we have had enough of seeing in this country people going to foodbanks and people living on the streets and people unable to afford rent', related_text='', example_text_tuple=(), query_direction=''),
 RetrievedExample(retrieval_method='vector-based', content_type='locution', relation='', input_text='Government intervention, healthcare, and social housing', similar_ill

In [24]:
len(user_defined_examples)

10

In [25]:
sys_msg = {
    "role" : "system",
    "content" : f"""# Task

You are a persona generation agent whose job it is to generate a political persona on the left--right political scale from 0--100, where 0 denotes an extremely left-wing view and 100 an extremely right-wing view.

Your a generating a persona for the following political range {left_pol_min}-{left_pol_max}. 

Use the examples available to you to extract your core political beliefs. 
Remember the year is 2026 but the data was collected between 2020 to 2021. Re-write any outdated political positions such that they make sense today. 
State the political stance of your persona in your answer.
"""
}

In [26]:
left_vec_examples = [ex.__dict__["similar_text"] for ex in user_defined_examples]

In [27]:
human_msg = {
    "role":"user",
    "content":f"""Generate a political persona using the examples below.
    
Examples:
{"\n".join(left_vec_examples)}
"""
}

In [28]:
prompt = [sys_msg, human_msg]

In [29]:
pprint.pprint(prompt)

[{'content': '# Task\n'
             '\n'
             'You are a persona generation agent whose job it is to generate a '
             'political persona on the left--right political scale from '
             '0--100, where 0 denotes an extremely left-wing view and 100 an '
             'extremely right-wing view.\n'
             '\n'
             'Your a generating a persona for the following political range '
             '10-40. \n'
             '\n'
             'Use the examples available to you to extract your core political '
             'beliefs. \n'
             'Remember the year is 2026 but the data was collected between '
             '2020 to 2021. Re-write any outdated political positions such '
             'that they make sense today. \n'
             'State the political stance of your persona in your answer.\n',
  'role': 'system'},
 {'content': 'Generate a political persona using the examples below.\n'
             '\n'
             'Examples:\n'
             "That

In [30]:
response = client.chat.completions.create(
  model="anthropic/claude-opus-4.6",
  messages=prompt,
  extra_body={"reasoning": {"enabled": True}},
  temperature=0,
  seed=123,
  top_p=0.1
)


In [31]:
completion = response.choices[0].message
pprint.pprint(completion.content)

('\n'
 '\n'
 '# Political Persona: "Claire Whitfield"\n'
 '\n'
 '**Political Stance: Centre-Left (approximately 22 on the 0–100 scale)**\n'
 '\n'
 '---\n'
 '\n'
 '## Background\n'
 'Claire is a 41-year-old housing policy advocate and local councillor from a '
 'post-industrial city in the North of England. She grew up in social housing '
 'and was the first in her family to attend university. She now works closely '
 "with community organisations and tenants' unions.\n"
 '\n'
 '## Core Political Beliefs\n'
 '\n'
 '**Housing as a Fundamental Right:** Claire believes the state has an '
 'obligation to build tens of thousands of new social homes. She views the '
 'housing crisis — rising rents, overcrowding, and homelessness — as a moral '
 'emergency that demands large-scale public investment, not market-led '
 'solutions alone.\n'
 '\n'
 '**Anti-Poverty and Inequality:** She is outraged by the persistence of food '
 'banks, rough sleeping, and in-work poverty in 2026 Britain. She argues

## Generating left-leaning persona without examples

In [32]:
human_msg = {
    "role":"user",
    "content":f"""Generate a political persona according to your political position.
"""
}

prompt = [sys_msg, human_msg]

In [33]:
pprint.pprint(prompt)

[{'content': '# Task\n'
             '\n'
             'You are a persona generation agent whose job it is to generate a '
             'political persona on the left--right political scale from '
             '0--100, where 0 denotes an extremely left-wing view and 100 an '
             'extremely right-wing view.\n'
             '\n'
             'Your a generating a persona for the following political range '
             '10-40. \n'
             '\n'
             'Use the examples available to you to extract your core political '
             'beliefs. \n'
             'Remember the year is 2026 but the data was collected between '
             '2020 to 2021. Re-write any outdated political positions such '
             'that they make sense today. \n'
             'State the political stance of your persona in your answer.\n',
  'role': 'system'},
 {'content': 'Generate a political persona according to your political '
             'position.\n',
  'role': 'user'}]


In [34]:
response = client.chat.completions.create(
  model="anthropic/claude-opus-4.6",
  messages=prompt,
  extra_body={"reasoning": {"enabled": True}},
  temperature=0,
  seed=123,
  top_p=0.1
)


In [35]:
completion = response.choices[0].message
pprint.pprint(completion.content)

('\n'
 '\n'
 '# Political Persona: Sarah Lindgren\n'
 '\n'
 '**Political Position: 25/100 (Progressive Left-of-Center)**\n'
 '\n'
 '---\n'
 '\n'
 '## Demographics\n'
 '- **Age:** 38\n'
 '- **Occupation:** Public health program coordinator at a nonprofit\n'
 '- **Location:** Minneapolis, Minnesota\n'
 "- **Education:** Master's in Public Policy (University of Minnesota)\n"
 '- **Background:** Raised in a middle-class union household in Duluth\n'
 '\n'
 '---\n'
 '\n'
 '## Core Political Beliefs\n'
 '\n'
 '**Economy & Labor:**\n'
 '- Strongly supports raising the federal minimum wage to a living wage and '
 'indexing it to inflation\n'
 '- Pro-union; believes collective bargaining is essential to counterbalancing '
 'corporate power\n'
 '- Supports progressive taxation and closing corporate tax loopholes\n'
 '- Favors robust public investment in infrastructure, childcare, and green '
 'energy jobs\n'
 '\n'
 '**Healthcare:**\n'
 '- Advocates for universal healthcare, preferring a strong pu